# 03 — Leakage-safe train / validation / test splits
**Level: intermediate.** A model cannot be evaluated fairly when the same scenario appears in both training and test data. We create frozen group-disjoint Parquet splits with hashes.

In [ ]:
import os
from pathlib import Path
import json
from nohack.datasets.prepare import prepare_dataset
source = Path(os.environ.get("NOHACK_DATASET", "data/interim/labeled_nohackflow.csv"))
out = Path(os.environ.get("NOHACK_SPLITS", "data/processed/nohack_splits"))
if not source.is_file():
    raise FileNotFoundError("NOHACK_DATASET must point to an approved NoHackFlow labeled table")
manifest = prepare_dataset(source, out, seed=42, val_fraction=0.20, test_fraction=0.20)
print(json.dumps(manifest, indent=2))

In [ ]:
import pandas as pd
train = pd.read_parquet(out / "train.parquet")
val = pd.read_parquet(out / "val.parquet")
test = pd.read_parquet(out / "test.parquet")
for name, frame in [("train", train), ("validation", val), ("test", test)]:
    print(name, len(frame), "rows;", frame.group_id.nunique(), "groups")
assert set(train.group_id).isdisjoint(val.group_id)
assert set(train.group_id).isdisjoint(test.group_id)
assert set(val.group_id).isdisjoint(test.group_id)
print("PASS: independent group IDs are disjoint")

### Why split before tuning?
Feature preprocessing, threshold choice and hyperparameter selection must not peek at the held-out test set. Real-world generalization must also be checked with a second independent time period or network, when available.